# 01 — Exploratory Data Analysis (IMDb)

This notebook explores the preprocessed IMDb sentiment dataset to understand class balance, review length patterns, token frequency structure, and vocabulary trade-offs. The goal is to justify key preprocessing choices, especially the maximum sequence length and vocabulary size, before model training.

In [ ]:
import pickle
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

sns.set_theme(style="whitegrid")

In [ ]:
processed_dir = Path("data/processed")
fig_dir = Path("report/figs")
fig_dir.mkdir(parents=True, exist_ok=True)

train = torch.load(processed_dir / "train.pt", map_location="cpu")
val = torch.load(processed_dir / "val.pt", map_location="cpu")
test = torch.load(processed_dir / "test.pt", map_location="cpu")
with open(processed_dir / "vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

print("Split sizes")
print(f"Train: {len(train['labels'])}")
print(f"Val:   {len(val['labels'])}")
print(f"Test:  {len(test['labels'])}")
print(f"Vocab size: {len(vocab)}")

In [ ]:
splits = {'Train': train, 'Val': val, 'Test': test}
rows = []
for split_name, data in splits.items():
    labels = data['labels'].numpy()
    counts = np.bincount(labels, minlength=2)
    total = counts.sum()
    rows.append({'split': split_name, 'label': 0, 'count': int(counts[0]), 'pct': counts[0] / total * 100})
    rows.append({'split': split_name, 'label': 1, 'count': int(counts[1]), 'pct': counts[1] / total * 100})
    print(f"{split_name}: label 0 = {counts[0]} ({counts[0] / total * 100:.2f}%), label 1 = {counts[1]} ({counts[1] / total * 100:.2f}%)")

df_class = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=df_class, x='split', y='count', hue='label', ax=ax)
ax.set_title('IMDb class distribution by split')
ax.set_xlabel('Split')
ax.set_ylabel('Count')
ax.legend(title='Label', labels=['0', '1'])
fig.tight_layout()
fig.savefig(fig_dir / 'eda_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
raw_lengths = train['raw_lengths'].numpy()
summary = {
    'min': int(raw_lengths.min()),
    'max': int(raw_lengths.max()),
    'mean': float(raw_lengths.mean()),
    'median': float(np.median(raw_lengths)),
    'p90': float(np.percentile(raw_lengths, 90)),
    'p95': float(np.percentile(raw_lengths, 95)),
}
print(pd.Series(summary).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(raw_lengths, bins=50, kde=False, ax=ax)
ax.axvline(256, color='red', linestyle='--', linewidth=2, label='max_len = 256')
ax.set_title('Distribution of raw review lengths in train split')
ax.set_xlabel('Raw token length')
ax.set_ylabel('Count')
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / 'eda_length_hist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
id_to_token = getattr(vocab, 'idx_to_token', None)
if id_to_token is None:
    id_to_token = {idx: token for token, idx in vocab.token_to_idx.items()}

counter = Counter()
for seq in train['input_ids']:
    for idx in seq.tolist():
        token = id_to_token.get(int(idx), '<unk>')
        if token not in {'<pad>', '<unk>'}:
            counter[token] += 1

top_tokens = counter.most_common(30)
top_df = pd.DataFrame(top_tokens, columns=['token', 'count']).sort_values('count', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=top_df, x='count', y='token', ax=ax, color='steelblue')
ax.set_title('Top-30 most frequent tokens after preprocessing')
ax.set_xlabel('Count')
ax.set_ylabel('Token')
fig.tight_layout()
fig.savefig(fig_dir / 'eda_top_tokens.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
train_sequences = [seq.tolist() for seq in train['input_ids']]
train_token_counts = Counter()
for seq in train_sequences:
    for idx in seq:
        token = id_to_token.get(int(idx), '<unk>')
        if token not in {'<pad>', '<unk>'}:
            train_token_counts[token] += 1

def oov_rate_for_top_n(n: int) -> float:
    top_tokens_n = {tok for tok, _ in train_token_counts.most_common(n)}
    total_occurrences = 0
    oov_occurrences = 0
    for seq in train_sequences:
        for idx in seq:
            token = id_to_token.get(int(idx), '<unk>')
            if token in {'<pad>', '<unk>'}:
                continue
            total_occurrences += 1
            if token not in top_tokens_n:
                oov_occurrences += 1
    return oov_occurrences / total_occurrences * 100 if total_occurrences else 0.0

oov_rows = []
for n in [10000, 20000, 30000]:
    oov_rows.append({'vocab_size': n, 'train_oov_rate_pct': oov_rate_for_top_n(n)})

oov_df = pd.DataFrame(oov_rows)
print(oov_df.to_string(index=False, formatters={'train_oov_rate_pct': '{:.2f}'.format}))
oov_df

## Summary

- The IMDb dataset is close to class-balanced across train, validation, and test splits.
- Review lengths have a long tail, and the histogram supports a max sequence length of 256 as a reasonable truncation point.
- The most frequent tokens are concentrated among common sentiment and function words after preprocessing.
- The OOV-rate comparison shows a clear trade-off between vocabulary size and token coverage, supporting `vocab_size = 20000` as a balanced choice.